# Revision: multi-category, multi-seed coreset matrix

논문 R3 실험용 artifact를 생성합니다. 기본 행렬은 MVTec AD 15개 category × 5개 coreset ratio × 3개 seed = 225개입니다. Google Drive의 검증 완료 bundle은 건너뛰고, 중단된 variant는 다시 생성합니다. Colab 세션당 `CATEGORIES_TO_RUN`을 1~3개로 줄여 실행하세요.


In [ ]:
from pathlib import Path
from decimal import Decimal
import json, shutil, subprocess, sys

REPOSITORY_URL = "https://github.com/minwoo1119/edge-anomaly-detection.git"
REPOSITORY_ROOT = Path("/content/edge-anomaly-detection")
DRIVE_INPUT_ROOT = Path("/content/drive/MyDrive/edge-anomaly-input")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/edge-anomaly-output/revision_coreset_r3")
LOCAL_WORK_ROOT = Path("/content/revision_coreset_r3")
ALL_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather",
    "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper",
]
CATEGORIES_TO_RUN = ALL_CATEGORIES[:2]  # 세션마다 1~3개로 바꾸세요.
SEEDS = [42, 123, 2026]
RATIOS = [0.01, 0.025, 0.05, 0.10, 0.20]
CALIBRATION_IMAGES = 100
AUTO_DOWNLOAD_DATASET = True

def ratio_tag(value):
    return "c" + format(Decimal(str(value)).normalize(), "f").replace(".", "")


In [ ]:
import torch
from google.colab import drive
drive.mount("/content/drive")
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0))

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPOSITORY_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPOSITORY_ROOT / "requirements.txt")], check=True)
GIT_COMMIT = subprocess.check_output(["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("commit:", GIT_COMMIT)


In [ ]:
local_dataset_root = REPOSITORY_ROOT / "datasets" / "MVTecAD"
for category in CATEGORIES_TO_RUN:
    if category not in ALL_CATEGORIES:
        raise ValueError(f"Unknown MVTec category: {category}")
    source = DRIVE_INPUT_ROOT / "datasets" / "MVTecAD" / category
    destination = local_dataset_root / category
    if not destination.is_dir() and source.is_dir():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination)
    if not destination.is_dir() and AUTO_DOWNLOAD_DATASET:
        local_dataset_root.mkdir(parents=True, exist_ok=True)
        code = "from anomalib.data import MVTecAD; " + f"d=MVTecAD(root={str(local_dataset_root)!r}, category={category!r}); d.prepare_data()"
        subprocess.run([sys.executable, "-c", code], check=True, cwd=REPOSITORY_ROOT)
    required = [destination / "train" / "good", destination / "test" / "good", destination / "ground_truth"]
    if not all(path.is_dir() for path in required):
        raise FileNotFoundError(f"Incomplete dataset for {category}: {required}")
    if not source.is_dir():
        source.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(destination, source)
        print("Cached dataset:", source)


In [ ]:
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

for category in CATEGORIES_TO_RUN:
    for seed in SEEDS:
        drive_seed_category = DRIVE_OUTPUT_ROOT / f"seed{seed}" / category
        missing = []
        for ratio in RATIOS:
            tag = ratio_tag(ratio)
            target = drive_seed_category / tag
            marker = target / "COMPLETE.json"
            bundle = target / f"{category}_artifacts.zip"
            checksum = bundle.with_suffix(bundle.suffix + ".sha256")
            if not (marker.is_file() and bundle.is_file() and checksum.is_file()):
                missing.append(ratio)
        if not missing:
            print(f"SKIP complete: {category} seed={seed}")
            continue

        local_seed_root = LOCAL_WORK_ROOT / f"seed{seed}"
        command = [
            sys.executable, "src/run_coreset_ablation.py",
            "--dataset-root", str(local_dataset_root), "--category", category,
            "--output-root", str(local_seed_root), "--ratios", *[str(value) for value in missing],
            "--calibration-images", str(CALIBRATION_IMAGES), "--accelerator", "gpu",
            "--devices", "1", "--seed", str(seed), "--resume",
        ]
        subprocess.run(command, check=True, cwd=REPOSITORY_ROOT)

        for ratio in missing:
            tag = ratio_tag(ratio)
            source_root = local_seed_root / category / tag
            source_bundle = source_root / "deployment" / f"{category}_artifacts.zip"
            source_checksum = source_bundle.with_suffix(source_bundle.suffix + ".sha256")
            source_marker = source_root / "COMPLETE.json"
            target = drive_seed_category / tag
            target.mkdir(parents=True, exist_ok=True)
            for source, name in ((source_bundle, source_bundle.name), (source_checksum, source_checksum.name), (source_marker, "COMPLETE.json")):
                partial = target / f".{name}.partial"
                shutil.copy2(source, partial)
                partial.replace(target / name)
            shutil.rmtree(source_root)
            print(f"COMPLETE: {category} seed={seed} ratio={ratio} -> {target}")


In [ ]:
missing = []
for category in ALL_CATEGORIES:
    for seed in SEEDS:
        for ratio in RATIOS:
            target = DRIVE_OUTPUT_ROOT / f"seed{seed}" / category / ratio_tag(ratio)
            bundle = target / f"{category}_artifacts.zip"
            if not ((target / "COMPLETE.json").is_file() and bundle.is_file() and bundle.with_suffix(bundle.suffix + ".sha256").is_file()):
                missing.append((category, seed, ratio))
print("completed:", len(ALL_CATEGORIES) * len(SEEDS) * len(RATIOS) - len(missing), "/", len(ALL_CATEGORIES) * len(SEEDS) * len(RATIOS))
print("missing:", missing[:30], "..." if len(missing) > 30 else "")
print("Drive output:", DRIVE_OUTPUT_ROOT)
